# Qwen3-VL-8B ChartQA — 微調前後對照評估（Colab A100）

**前置**：Phase 2 訓練已完成、LoRA adapter 已在 HF Hub。

**用法**：
1. Colab → 檔案 → 上傳筆記本 → 選這個檔案；執行階段選 **A100 GPU**
2. 左側 🔑 Secrets 確認 `HF_TOKEN` 可用
3. 建議先 `EVAL_LIMIT = 100` 快跑一輪確認 pipeline 沒問題，再改 `None` 跑完整 test split（2500 筆 × 2 個模型）
4. 跑完記得：執行階段 → 中斷連線並刪除執行階段

**評估設計**：
- ChartQA test split：**human**（1250 筆，人寫問題）與 **augmented/machine**（1250 筆，機器產生問題）分開計分
- 指標：**relaxed accuracy**（數值答案容許 5% 誤差，其餘不分大小寫精確匹配）— 與 ChartQA 論文/lmms-eval 同一實作
- baseline 用訓練起點同一個 4-bit 底模（`unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit`），與微調版（同底模 + LoRA）比較才公平（兩邊量化誤差相同）
- 產物 push 到 adapter repo 的 `eval/`：`results.json`、完整預測 `predictions_*.json`、10 個案例圖 `cases/`


In [ ]:
# 1. GPU 檢查
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

In [ ]:
%%capture
# 2. 安裝相依（與訓練 notebook 相同）
import os, re
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"  # 先關閉：加速下載器在部分網路環境會靜默卡死；改回 "1" 前先確認這次全程沒卡住
os.environ["HF_HUB_DISABLE_XET"] = "1"  # Colab/GCP 對 hf_xet 傳輸有已知卡死問題（huggingface_hub #3266, #4085）
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.1
!pip install --no-deps trl==0.22.2
!pip uninstall -y -q hf_xet  # 雙保險：直接移除套件，避免 huggingface_hub 選用 Xet 傳輸而卡死

In [ ]:
# 3. HF 登入
from google.colab import userdata
from huggingface_hub import login, whoami
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
HF_USER = whoami()["name"]
print("logged in as:", HF_USER)

def _retry_hf(fn, *args, max_retries=5, base_delay=10.0, **kwargs):
    # HF Hub 的 Xet CDN 簽章在 Colab 上會間歇性失效，表現方式很多種（403、被包裝成
    # 「檔案不存在」的 OSError、離線載入殘缺快取的 AttributeError）。這裡單純重試；
    # 不能塞 force_download —— unsloth 內部是「先預下載、再離線載入」，離線階段
    # 收到 force_download 會直接 ValueError
    import time

    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            if attempt == max_retries:
                raise
            wait = base_delay * attempt
            print(f"  [HF retry] attempt {attempt}/{max_retries} failed ({type(e).__name__}: {e}); retrying in {wait:.0f}s")
            time.sleep(wait)

def _prefetch_repo(repo_id, repo_type="model", max_retries=6, base_delay=10.0):
    # 真正的關鍵：先用整檔下載把 repo 抓齊進本機快取（走已證實可靠的路徑，失敗自動重試）。
    # unsloth/transformers 的「預下載失敗 -> 退回離線載入」流程在快取不完整時會炸出
    # 各種怪錯（checkpoint_files None / 檔案不存在）；快取抓齊之後離線載入必定成功
    import time
    from huggingface_hub import snapshot_download

    for attempt in range(1, max_retries + 1):
        try:
            return snapshot_download(repo_id, repo_type=repo_type)
        except Exception as e:
            if attempt == max_retries:
                raise
            wait = base_delay * attempt
            print(f"  [prefetch {repo_id}] attempt {attempt}/{max_retries} failed ({type(e).__name__}); retrying in {wait:.0f}s")
            time.sleep(wait)

def _prefetch_model_and_base(repo_id):
    # LoRA adapter repo 會連同 adapter_config.json 指到的底模一起預抓
    import json, os
    path = _prefetch_repo(repo_id)
    cfg = os.path.join(path, "adapter_config.json")
    if os.path.exists(cfg):
        base = json.load(open(cfg)).get("base_model_name_or_path")
        if base:
            print(f"[prefetch] adapter 底模: {base}")
            _prefetch_repo(base)
    return path

In [ ]:
# 4. 設定
EVAL_LIMIT = 100          # <<< 先 100 快跑驗證 pipeline；OK 後改 None 跑完整 2500 筆
BATCH_SIZE = 8            # OOM 就降
MAX_NEW_TOKENS = 32
SEED = 3407

BASE_MODEL   = "unsloth/Qwen3-VL-8B-Instruct-unsloth-bnb-4bit"
ADAPTER_REPO = f"{HF_USER}/qwen3vl-8b-chartqa-lora"   # Phase 2 全量訓練的 adapter
print(f"{ADAPTER_REPO=}  {EVAL_LIMIT=}")

In [ ]:
# 5. ChartQA 載入與 prompt（與 src/chartqa_data.py 同步）
from datasets import Dataset

DATASET_ID = "HuggingFaceM4/ChartQA"
ANSWER_INSTRUCTION = "Answer the question using a single word or phrase."
HUMAN, MACHINE = 0, 1

def _download_with_retry(filename, max_retries=6, base_delay=5.0):
    # HF 的 Xet CDN 簽章在 Colab 上會間歇性失效（同一檔案這次成功、下次 403），
    # 不是檔案本身壞掉；重新請求會拿到新的簽章 URL，重試就會過
    import time
    from huggingface_hub import hf_hub_download
    from huggingface_hub.errors import HfHubHTTPError

    for attempt in range(1, max_retries + 1):
        try:
            return hf_hub_download(DATASET_ID, filename, repo_type="dataset")
        except HfHubHTTPError as e:
            if attempt == max_retries:
                raise
            wait = base_delay * attempt
            print(f"  [{filename}] attempt {attempt}/{max_retries} failed ({type(e).__name__}); retrying in {wait:.0f}s")
            time.sleep(wait)

def load_chartqa(split, n=None, human_or_machine=None, seed=42):
    # 直接抓該 split 的 parquet 檔（整檔循序下載＋失敗重試），不用 datasets.load_dataset()：
    # 它的 eager 模式會連其他 split 一起準備，streaming 模式則用 byte-range 讀取，
    # 都撞過 HF Xet CDN 在 Colab 上簽章失效的 403（結果證實是間歇性的，不是特定檔案壞掉）
    from huggingface_hub import HfApi

    files = sorted(f for f in HfApi().list_repo_files(DATASET_ID, repo_type="dataset")
                   if f.startswith(f"data/{split}-"))
    local_paths = [_download_with_retry(f) for f in files]
    ds = Dataset.from_parquet(local_paths)
    if human_or_machine is not None:
        ds = ds.filter(lambda ex: ex["human_or_machine"] == human_or_machine)
    if n is not None and n < len(ds):
        ds = ds.shuffle(seed=seed).select(range(n))
    return ds

def get_answer(example):
    label = example["label"]
    return str(label[0]) if isinstance(label, list) else str(label)

def to_messages(example, include_answer=True):
    user_content = [
        {"type": "image", "image": example["image"]},
        {"type": "text", "text": f"{example['query']}\n{ANSWER_INSTRUCTION}"},
    ]
    messages = [{"role": "user", "content": user_content}]
    if include_answer:
        messages.append({"role": "assistant",
                         "content": [{"type": "text", "text": get_answer(example)}]})
    return messages

In [ ]:
# 6. relaxed accuracy（與 src/relaxed_accuracy.py 同步；ChartQA 論文原版邏輯）
def _to_float(text):
    try:
        if text.endswith("%"):
            return float(text.rstrip("%")) / 100.0
        return float(text)
    except ValueError:
        return None

def relaxed_correctness(prediction, target, max_relative_change=0.05):
    prediction_float = _to_float(prediction)
    target_float = _to_float(target)
    if prediction_float is not None and target_float:
        relative_change = abs(prediction_float - target_float) / abs(target_float)
        return relative_change <= max_relative_change
    return prediction.lower() == target.lower()

def normalize_prediction(text):
    text = text.strip()
    if text.endswith("."):
        text = text[:-1].rstrip()
    return text

def relaxed_accuracy(predictions, targets):
    assert len(predictions) == len(targets)
    if not predictions:
        return 0.0
    return sum(relaxed_correctness(normalize_prediction(p), t)
               for p, t in zip(predictions, targets)) / len(predictions)

In [ ]:
# 7. 批次評估函式（同一函式跑 baseline 與微調版）
import torch
from tqdm.auto import tqdm

def run_eval(model, tokenizer, ds, batch_size=BATCH_SIZE, max_new_tokens=MAX_NEW_TOKENS):
    from unsloth import FastVisionModel
    FastVisionModel.for_inference(model)
    inner = getattr(tokenizer, "tokenizer", tokenizer)
    inner.padding_side = "left"       # decoder-only 批次生成必須左 padding
    preds = []
    for i in tqdm(range(0, len(ds), batch_size)):
        batch = [ds[j] for j in range(i, min(i + batch_size, len(ds)))]
        texts = [tokenizer.apply_chat_template(
                    to_messages(ex, include_answer=False),
                    add_generation_prompt=True, tokenize=False) for ex in batch]
        images = [ex["image"].convert("RGB") for ex in batch]
        inputs = tokenizer(images=images, text=texts, padding=True,
                           return_tensors="pt").to("cuda")
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        gen = out[:, inputs["input_ids"].shape[1]:]
        preds += [t.strip() for t in inner.batch_decode(gen, skip_special_tokens=True)]
    return preds

def eval_both_splits(model, tokenizer, tag):
    results = {}
    for name, hom in [("human", HUMAN), ("augmented", MACHINE)]:
        ds = load_chartqa("test", n=EVAL_LIMIT, human_or_machine=hom, seed=SEED)
        golds = [get_answer(ex) for ex in ds]
        queries = [ex["query"] for ex in ds]
        preds = run_eval(model, tokenizer, ds)
        acc = relaxed_accuracy(preds, golds)
        results[name] = {"n": len(ds), "relaxed_accuracy": acc,
                         "queries": queries, "predictions": preds, "golds": golds}
        print(f"[{tag}] {name}: relaxed_accuracy = {acc:.4f}  (n={len(ds)})")
    return results

In [ ]:
# 8. 評估 baseline（微調前）
from unsloth import FastVisionModel

_prefetch_model_and_base(BASE_MODEL)
base_model, base_tok = _retry_hf(FastVisionModel.from_pretrained, BASE_MODEL, load_in_4bit=True)
baseline_results = eval_both_splits(base_model, base_tok, "baseline")

del base_model
torch.cuda.empty_cache()

In [ ]:
# 9. 評估微調版（同一底模 + LoRA adapter）
_prefetch_model_and_base(ADAPTER_REPO)
ft_model, ft_tok = _retry_hf(FastVisionModel.from_pretrained, ADAPTER_REPO, load_in_4bit=True)
finetuned_results = eval_both_splits(ft_model, ft_tok, "finetuned")

In [ ]:
# 10. 對照表 + 結果 push 到 HF Hub
import json
import pandas as pd
from huggingface_hub import HfApi

rows = []
for split in ["human", "augmented"]:
    rows.append({
        "test split": split,
        "n": baseline_results[split]["n"],
        "baseline (before)": round(baseline_results[split]["relaxed_accuracy"], 4),
        "fine-tuned (after)": round(finetuned_results[split]["relaxed_accuracy"], 4),
        "Δ": round(finetuned_results[split]["relaxed_accuracy"]
                   - baseline_results[split]["relaxed_accuracy"], 4),
    })
overall_b = sum(baseline_results[s]["relaxed_accuracy"] * baseline_results[s]["n"] for s in baseline_results) /             sum(baseline_results[s]["n"] for s in baseline_results)
overall_f = sum(finetuned_results[s]["relaxed_accuracy"] * finetuned_results[s]["n"] for s in finetuned_results) /             sum(finetuned_results[s]["n"] for s in finetuned_results)
rows.append({"test split": "overall", "n": sum(baseline_results[s]["n"] for s in baseline_results),
             "baseline (before)": round(overall_b, 4), "fine-tuned (after)": round(overall_f, 4),
             "Δ": round(overall_f - overall_b, 4)})
df = pd.DataFrame(rows)
print(df.to_markdown(index=False))

os.makedirs("eval_out", exist_ok=True)
summary = {"base_model": BASE_MODEL, "adapter": ADAPTER_REPO, "eval_limit": EVAL_LIMIT,
           "metric": "relaxed_accuracy(5%)", "table": rows}
with open("eval_out/results.json", "w") as f:
    json.dump(summary, f, indent=1)
for tag, res in [("baseline", baseline_results), ("finetuned", finetuned_results)]:
    slim = {s: {k: v for k, v in res[s].items()} for s in res}
    with open(f"eval_out/predictions_{tag}.json", "w") as f:
        json.dump(slim, f, indent=1)

api = HfApi()
api.upload_folder(folder_path="eval_out", path_in_repo="eval", repo_id=ADAPTER_REPO)
print(f"pushed -> https://huggingface.co/{ADAPTER_REPO}/tree/main/eval")

In [ ]:
# 11. 10 個具體案例（圖 + 問題 + 微調前後回答）
# 從 human split 挑：5 個「微調後才答對」+ 3 個「兩者都錯」+ 2 個「兩者都對」
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import textwrap

split = "human"
ds_h = load_chartqa("test", n=EVAL_LIMIT, human_or_machine=HUMAN, seed=SEED)
b = baseline_results[split]; f_ = finetuned_results[split]

def ok(pred, gold):
    return relaxed_correctness(normalize_prediction(pred), gold)

improved = [i for i in range(len(ds_h))
            if not ok(b["predictions"][i], b["golds"][i]) and ok(f_["predictions"][i], f_["golds"][i])]
both_bad = [i for i in range(len(ds_h))
            if not ok(b["predictions"][i], b["golds"][i]) and not ok(f_["predictions"][i], f_["golds"][i])]
both_good = [i for i in range(len(ds_h))
             if ok(b["predictions"][i], b["golds"][i]) and ok(f_["predictions"][i], f_["golds"][i])]
picks = improved[:5] + both_bad[:3] + both_good[:2]
print(f"improved={len(improved)}  both_bad={len(both_bad)}  both_good={len(both_good)}  -> picked {len(picks)}")

os.makedirs("eval_out/cases", exist_ok=True)
cases_meta = []
for rank, i in enumerate(picks, 1):
    ex = ds_h[i]
    fig, ax = plt.subplots(figsize=(7, 8))
    ax.imshow(ex["image"]); ax.axis("off")
    q = "\n".join(textwrap.wrap("Q: " + ex["query"], 80))
    cap = (f"{q}\n"
           f"gold: {b['golds'][i]}\n"
           f"before: {b['predictions'][i]}   [{'O' if ok(b['predictions'][i], b['golds'][i]) else 'X'}]\n"
           f"after : {f_['predictions'][i]}   [{'O' if ok(f_['predictions'][i], f_['golds'][i]) else 'X'}]")
    ax.set_title(cap, fontsize=9, loc="left", family="monospace")
    fig.tight_layout()
    path = f"eval_out/cases/case_{rank:02d}.png"
    fig.savefig(path, dpi=130, bbox_inches="tight"); plt.close(fig)
    cases_meta.append({"rank": rank, "index": i, "query": ex["query"], "gold": b["golds"][i],
                       "before": b["predictions"][i], "after": f_["predictions"][i]})

with open("eval_out/cases/cases.json", "w") as fp:
    json.dump(cases_meta, fp, indent=1)
api.upload_folder(folder_path="eval_out/cases", path_in_repo="eval/cases", repo_id=ADAPTER_REPO)
print(f"pushed -> https://huggingface.co/{ADAPTER_REPO}/tree/main/eval/cases")

## 下一步

- `EVAL_LIMIT = 100` 的快跑數字只是 pipeline 驗證，**正式對照表要用 `EVAL_LIMIT = None`（完整 2500 筆）重跑**
- 對 baseline 分數做 sanity check：Qwen3-VL-8B 級別模型 ChartQA 官方報告約在 80% 上下（4-bit 會略低）；若 baseline 低太多（例如 <60%），先檢查 prompt/解析而不是急著怪模型
- 跑完記錄第 10 格的對照表，接著執行 Phase 4 量化 + vLLM benchmark
- 跑完記得：執行階段 → **中斷連線並刪除執行階段**
